Project Structure
----

1. Introduction
2. Install Libraries
3. Import Libraries
4. Load Embedding Model
5. Understanding Tokenization
6. Converting Tokens to IDs
7. Creating Input Tensors
8. Running BERT
9. Exploring Embeddings
10. Similarity Experiments
11. Visualizations
12. Challenges & Exercises

  Learning Objectives
  ---
  1. What exactly is an embedding?
  2. Why are embeddings vectors?
  3. Why does BERT produce one vector per token?
  4. What is the [CLS] token?
  5. Why do embeddings have 768 dimensions?
  6. What is an attention mask?
  7. Why does "bank" have different embeddings in different contexts?
  8. How do we compare embeddings?

# Part 1: Introduction to Embeddings


----
# Topic: What is embedding?
----
The problem

- Imagine we have three words : Cat, Dog, Car
- A computer does not understand these words, if we simply assign ids to each word: [1] Cat | [2] Dog |
[3] Car
- We can see that using the assigned ids that Dog is closer to Cat compared to Car. But that is just because of the numbering, it does not reflect any significant meaning.
- Now imagine: [500] Apple | [10] Orange | [20] Laptop, a model would incorrectly infer that Orange is more similar to Laptop than Apple, simply because of the IDs and this is why IDs are identifiers not representations.




One-Hot Encoding

- Before embeddings, NLP often used one-hto vectors. Suppose our vocabulary is: Cat, Dog, Car, House, Tree
- Then: Cat [1,0,0,0,0] | Dog [0,1,0,0,0] | Car [0,0,1,0,0]
- One can notice that every word is equally distant from every other word. But still the model cannot tell that: Cat and Dog are animals, Car and Bus are vehicles.
- One-hot encoding completely ignores sematic relationships.

Solution - Embeddings

- Instead of assigning IDs or one-hot vectors, we learn dense vectors that capture meaning.
- An embedding for Cat might look like: [0.23, -0.11, 0.92, 0.44, ...]
- For Dog: [0.20, -0.09, 0.88, 0.39, ...]
- For Car: [-0.72, 0.51, -0.13, 0.80, ...]
- One can notice that Cat and Dog have similar values than Cat and Car. Models learn these vectors from large amounts of text so that similar meanings end up close together in the cector space.


What is BERT Actually Producing

- BERT does not output a single number or a single label. Instead, it converts each token into  a high-dimensional vector.
- For example: I love AI
- Becomes something conceptually like: I [767 numbers] | love [767 numbers] | AI [767 numbers]
- Each token gets its won represenationtion, and that representation depends on the context.

---

In [1]:
# Install the required libraries
!pip install -q transformers torch sentence-transformers scikit-learn matplotlib pandas

In [2]:
# Import required libraries
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from transformers import BertTokenizer, BertModel

In [3]:
# Load the tokenizer and model
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
# Input text
text = "I love studying Artificial Intelligence."

# Tokenize without special tokens
tokens = tokenizer.tokenize(text)
print(tokens)

['i', 'love', 'studying', 'artificial', 'intelligence', '.']


Notice the following:

- Everything is lowercase, that is because of we are using "bert-base-uncased". There is alos a cased version of BERT that preserves capitalization.
- Punctuation is its own token, instead of 'intelligence.', BERT produces 'intelligence' '.'
- Theperiod carries information about sentence boundaries, so it becomes its own token.

In [5]:
# Tokenize with special tokens
encoding = tokenizer(text)
encoding

{'input_ids': [101, 1045, 2293, 5702, 7976, 4454, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}

In [7]:
# Decode IDs back into tokens
tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"])
print(tokens)

['[CLS]', 'i', 'love', 'studying', 'artificial', 'intelligence', '.', '[SEP]']


In [8]:
# Print number of tokens
print(f'Number of tokens: {len(tokens)}')

Number of tokens: 8


The model processes 8 tokens, not 6.

Understanding special characters : [CLS] and [SEP]

- The first token is always: [CLS]
- Think of it as "Start porcessing a new sequence.". Its final embedding is often used as a represenatation of the entire sentence for tasks like classification.

- The last token is: [SEP]
- It marks the end of a sentence or separates two sentences.
- For example: Sentence A [SEP] Sentence B
- This is useful for tasks such as question answering or natural language inference.


Neural networks do not process words, they process numbers.

In [9]:
# Convert tokens to IDs
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print(token_ids)

[101, 1045, 2293, 5702, 7976, 4454, 1012, 102]


- Each number is simply an index into BERT's vocabulary
- NB: These IDs do not encode meaning. They are lookup keys used to retrieve learned embedding vectors from BERT's embedding matrix.

In [10]:
# Verify that the mapping is reversible, decode back
decoded = tokenizer.convert_ids_to_tokens(token_ids)
print(decoded)

['[CLS]', 'i', 'love', 'studying', 'artificial', 'intelligence', '.', '[SEP]']


Experiment with WordPiece
- Now let's use a words that BERT may split into subwords.

In [12]:
words = [
    "unbelievably",
    "playing",
    "playground",
    "electromagnetism",
    "transformers",
    "ChatGPT",
    "Johannesburg"
]

for word in words:
  print(f'{word:20} -> {tokenizer.tokenize(word)}')

unbelievably         -> ['un', '##bel', '##ie', '##va', '##bly']
playing              -> ['playing']
playground           -> ['playground']
electromagnetism     -> ['electro', '##ma', '##gne', '##tism']
transformers         -> ['transformers']
ChatGPT              -> ['chat', '##gp', '##t']
Johannesburg         -> ['johannesburg']


- The ## prefix means this is a continuation of the previous token.
- This allows BERT to represent rare or unseen words by combining known subword pieces.